# M4 — Atribución ATT&CK por evento (multilabel con LogReg)

> **El problema, en palabras simples.**
>
> Clasificar el tier (benigno, sospechoso, malicioso) ordena la cola, pero el analista necesita saber además **qué técnica de ataque es**: ¿explotación pública (T1190), exfiltración (T1041), impacto (T1486)? Este cuaderno pregunta si un modelo lineal puede predecir las técnicas ATT&CK de cada evento, en vez del tier.
>
> La respuesta, adelantada: **a nivel evento solo T1190 es separable; la cola no tiene de dónde aprender**. La cabeza es casi determinista del tier y la cola es larguísima. El experimento lo mide con honestidad para no prometer en la demo lo que el modelo no sabe.

> **Síntesis ejecutiva.**
>
> Solo filas threat (sospechoso + malicioso, 114 521; el benigno no porta técnicas y M1 ya lo filtra). Targets: 6 técnicas [T1190, T1041, T1567, T1204, T1486, T1566]. Partición temporal 80/20 sobre threat. Modelo `OneVsRest(LogisticRegression)` sobre TF-IDF 5k + tabular P1–P3 (receta de M1/M2, no se reentrena nada del cuaderno 1), más baseline oráculo-tier (predice técnicas desde el tier real).
>
> Resultado (`models/m4_metrics.json`): micro-F1 0.7484, macro-F1 0.2516, hamming 0.1162, subset-acc 0.6451 (oráculo: 0.9994/0.4997). T1190 perfecto y trivial; T1041/T1567 con F1 0.10; cola en cero. n_train 91 617, n_test 22 904.

| Técnica | Soporte live-threat | Qué mide |
|---|---|---|
| T1190 | 114 507 (todo threat) | Cabeza trivial: está en todos |
| T1041 | 7 742 | Cabeza que el lineal no separa |
| T1567 | 7 728 (= todos los malicious) | Casi determinista del tier |
| T1204 | 20 (10 en test) | Cola: el modelo los atrapa a ciegas |
| T1486, T1566 | 0 en live (solo histórico) | Sin soporte: F1 0 estructural |

> **Fuentes normativas.**
>
> EDA §§8–9 (`notebooks/eda_precinct6.ipynb`), `models/m4_metrics.json`, `models/m4b_metrics.json` y `models/m4c_exps.json` (familia M4), `reports/paper.pdf` §Limitations (ablación multilabel).
>
> **Reproducción:** `uv run jupyter nbconvert --to notebook --execute notebooks/m4_multilabel_attack.ipynb --output m4_multilabel_attack.ipynb`.

---

**Convención de lectura.**

Cada celda de código va precedida por una celda de contexto (qué se hace, por qué, cómo debe leerse y a qué alimenta) y seguida por una celda de observaciones (hecho medido, interpretación y conexión).

La sección final integra los hallazgos y fija el veredicto para la demo.



### 1. Por qué existe esta atribución y cómo se va a evaluar

**1.1 Problema.**

El triage (M1→M2→M3) decide *escalar o monitorear*. La atribución decide *con qué playbook responder*. Son tareas distintas: la segunda es multilabel (un evento porta varias técnicas) sobre un vocabulario con cabeza determinista y cola sin soporte.

**1.2 Hipótesis.**

- **H4a. La cabeza es trivial o inalcanzable para el lineal.** T1190 saldrá perfecto por prevalencia (todo threat lo porta); T1041/T1567 no se separarán de sospechoso con TF-IDF + tabular, porque a nivel evento son firma de tier, no de texto.

- **H4b. La cola es estructuralmente cero.** T1486/T1566 sin soporte en live no pueden aprenderse; T1204 se atrapará solo a base de falsos positivos. Ningún hiperparámetro cambia esto: es escasez, no capacidad.

**1.3 Protocolo.**

1. **Población threat únicamente.** Benigno excluido por diseño (no porta técnicas; M1 ya lo filtra en cascada). Partición temporal 80/20 por cuantil sobre threat, coherente con la familia M4 (M4b/M4c/M4d usan el mismo corte; M1/M2/M3 usan ventanas UTC fijas sobre toda la población: se declara la diferencia).

2. **Métricas.** Micro-F1 y macro-F1 juntos (reportar solo micro mentiría: lo domina T1190), hamming loss y subset accuracy, más PR-AUC por técnica. Baseline oráculo-tier para calibrar: suspicious→[T1190], malicious→[T1190, T1041, T1567].

3. **Reutilización, no rueda nueva.** Features P1–P3 y TF-IDF 5k idénticos a M1/M2; nada del cuaderno 1 se reentrena. Lo único nuevo es Y (multilabel) y el oráculo.

**1.4 Criterio de veredicto.**

Si macro-F1 ronda 0.25 con cola en cero, la atribución por evento no se despliega: la consola servirá correspondencia tier→técnicas declarada como tal.



### 2. Carga y parseo: población threat y lista de técnicas

**2.1 Qué se hace.**

Se cargan las columnas necesarias de `signals.parquet`, se filtra la población threat (etiqueta distinta de benigno, 114 521 filas) y se parsean los arreglos JSON-string de `attack_techniques` restringidos a las 6 técnicas objetivo, con conteo de prevalencia y de etiquetas por evento.

**2.2 Por qué.**

El multilabel por evento solo tiene sentido donde hay etiquetas: el benigno porta lista vacía y añadirlo solo inflaría el micro-F1 con verdaderos negativos triviales. El conteo previo fija el techo antes de entrenar: cabeza masiva, cola ausente.

**2.3 Cómo debe leerse.**

Tamaño de threat, reparto sospechoso/malicioso, prevalencia por técnica en live y distribución de etiquetas por evento (sospechoso 1 etiqueta, malicioso 3, con 20 filas con T1204 adicional).

**2.4 Trazabilidad.**

Define la población y el vocabulario Y de todo el cuaderno. Los conteos anticipan H4a/H4b en forma falsable.



In [1]:
import json
from collections import Counter
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
ROOT = ROOT if (ROOT / 'pyproject.toml').exists() else ROOT.parent
print('ROOT:', ROOT)

COLS = ['label_binary','timestamp','event_time','message_sanitized','attack_techniques',
        'message_type','stream_name','product_name','vendor_name','pipeline',
        'org_id','action','severity','protocol']
df = pd.read_parquet(ROOT / 'data' / 'signals' / 'signals.parquet', columns=COLS)
th = df[df.label_binary != 'benign'].copy().reset_index(drop=True)
print('threat:', th.shape, th.label_binary.value_counts().to_dict())

TECHS = ['T1190','T1041','T1567','T1204','T1486','T1566']
def parse(v):
    try:
        t = json.loads(v) if v else []
        return [x for x in t if x in TECHS]
    except Exception:
        return []
th['tech_list'] = th.attack_techniques.fillna('').map(parse)
c = Counter(t for ts in th.tech_list for t in ts)
print('tecnicas en live-threat:', {k: c.get(k,0) for k in TECHS})
print('n-labels por evento:', th.tech_list.map(len).value_counts().to_dict())

ROOT: C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto


threat: (114521, 14) {'suspicious': 106793, 'malicious': 7728}


tecnicas en live-threat: {'T1190': 114507, 'T1041': 7742, 'T1567': 7728, 'T1204': 20, 'T1486': 0, 'T1566': 0}
n-labels por evento: {1: 106793, 3: 7708, 4: 20}


#### 2.1 Observaciones e insights

**Observación.**

Threat 114 521 filas. T1190 en 114 507 (prácticamente todo threat), T1041 en 7 742, T1567 en 7 728 (coincide con los maliciosos), T1204 en 20. Sospechoso porta 1 etiqueta, malicioso 3.

**Interpretación.**

La cabeza es casi función determinista del tier (T1567 equivale a malicioso) y T1190 es constante: ningún modelo puede extraer de ahí más información que la que ya da el tier. La cola (T1204/T1486/T1566) carece de masa crítica.

**Conexión.**

Plantea H4a/H4b con números: T1190 saldrá perfecto sin medir nada; T1041/T1567 exigirán separar malicioso de sospechoso a nivel evento, tarea que M2 ya mostró difícil fuera del texto de exploit.



### 3. Features P1–P3 y partición temporal sobre threat

**3.1 Qué se hace.**

Se derivan `hour`, `msg_len` y `lag_h` (receta P1–P3 idéntica a M1), se binariza Y con `MultiLabelBinarizer` sobre las 6 técnicas y se corta train/test en el cuantil 80 del timestamp (91 617/22 904).

**3.2 Por qué.**

Reutilizar P1–P3 sin cambios aísla el efecto de la variable respuesta: mismo X que M1/M2, distinto Y. Si el multilabel fracasa con el mismo X que ordena bien el tier, la conclusión es sobre la tarea, no sobre los features.

**3.3 Cómo debe leerse.**

Tamaños train/test, fecha de corte y densidad de Y (baja: la mayoría de celdas son cero). El corte por cuantil sobre threat se declara distinto de las ventanas UTC fijas de M1/M2/M3.

**3.4 Trazabilidad.**

Fija X/Y y el split de las secciones 4–6. La densidad anticipa hamming bajo por trivialidad (acertar ceros), no por pericia.



In [2]:
# features P1-P3 (igual que M1) + split temporal 80/20
CATS = ['message_type','stream_name','product_name','vendor_name','pipeline',
        'org_id','action','severity','protocol']
th['ts'] = pd.to_datetime(th['timestamp'], unit='s', utc=True)
th['hour'] = th.ts.dt.hour
th['msg_len'] = th.message_sanitized.fillna('').str.len()
th['lag_h'] = ((th['timestamp'] - th['event_time']) / 3600).clip(-1e4, 1e4).fillna(-1)
for col in CATS:
    th[col] = th[col].fillna('').astype(str)
msg = th.message_sanitized.fillna('')
cutoff = th['timestamp'].quantile(0.8)
tr = (th['timestamp'] <= cutoff).to_numpy(); te = ~tr
print(f'train/test: {tr.sum()}/{te.sum()}  corte: {pd.to_datetime(cutoff, unit="s", utc=True)}')

from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer(classes=TECHS)
Y = mlb.fit_transform(th.tech_list)
print('Y:', Y.shape, 'densidad:', round(Y.mean(), 4))

train/test: 91617/22904  corte: 2024-07-27 05:13:54.964786053+00:00


Y: (114521, 6) densidad: 0.1892


#### 3.1 Observaciones e insights

**Observación.**

Train 91 617, test 22 904, densidad de Y baja (matriz dispersa dominada por ceros y por la columna T1190).

**Interpretación.**

Con esta densidad, el hamming loss será bajo aunque el modelo no aprenda nada (predecir casi todo cero ya acierta). Por eso el hamming nunca se reporta solo: manda el par micro/macro-F1.

**Conexión.**

Habilita la matriz de la sección 4 y el entrenamiento de la 5. El corte declarado permite comparar dentro de la familia M4 (mismo protocolo que M4b/M4c/M4d).



### 4. Matriz TF-IDF 5k más tabular (ajuste solo en train)

**4.1 Qué se hace.**

TF-IDF 5k (unigramas+bigramas, sublineal, `min_df=5`) ajustado solo en train sobre `message_sanitized`, más one-hot podado top-30 y numéricas en passthrough; concatenación dispersa train/test.

**4.2 Por qué.**

Misma X textual+tabular de M2 con otro Y: si el texto separaba el tier malicioso (M2, recall 0.97), aquí se prueba si además separa técnicas dentro del threat. El ajuste exclusivo en train evita fuga del vocabulario futuro.

**4.3 Cómo debe leerse.**

Dimensiones de X train/test (ancho dominado por el vocabulario TF-IDF). Verificar que el vectorizador solo vio train.

**4.4 Trazabilidad.**

Es la entrada del OneVsRest y del oráculo. Reutiliza P4 sin reestimar nada del cuaderno 1.



In [3]:
# X: TF-IDF 5k (fit solo train) + tabular one-hot
from scipy.sparse import hstack
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder

vec = TfidfVectorizer(max_features=5000, ngram_range=(1,2), sublinear_tf=True, min_df=5)
Ttr = vec.fit_transform(msg[tr].tolist()); Tte = vec.transform(msg[te].tolist())
tab = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore', max_categories=30), CATS),
                           ('num', 'passthrough', ['hour','msg_len','lag_h'])])
Atr = tab.fit_transform(th[tr][CATS+['hour','msg_len','lag_h']])
Ate = tab.transform(th[te][CATS+['hour','msg_len','lag_h']])
Xtr = hstack([Ttr, Atr]).tocsr(); Xte = hstack([Tte, Ate]).tocsr()
print('X:', Xtr.shape, Xte.shape)

X: (91617, 5054) (22904, 5054)


#### 4.1 Observaciones e insights

**Observación.**

Matriz dispersa train/test con ancho ≈ 5k términos más categóricas podadas y 3 numéricas; vectorizador ajustado solo en train.

**Interpretación.**

El diseño es limpio contra fuga: el test futuro no aportó vocabulario. Todo gap posterior entre train y test será generalización real, incluida la deriva del 07-30.

**Conexión.**

Deja el terreno listo para el contraste modelo vs oráculo de la sección 5.



### 5. Modelo OneVsRest-LogReg más baseline oráculo-tier

**5.1 Qué se hace.**

Se entrena `OneVsRest(LogisticRegression)` (C=0.5, 1000 iteraciones, balanceada) sobre X, se predicen etiquetas duras en test y se construye el oráculo-tier (sospechoso→[T1190], malicioso→[T1190, T1041, T1567]) como techo de referencia.

**5.2 Por qué.**

Binary relevance (un clasificador por etiqueta, Tsoumakas y Katakis) es el baseline estándar del multilabel: ignora correlaciones entre etiquetas a cambio de simplicidad y costo lineal. El oráculo mide cuánta información porta ya el tier: si el modelo no supera al oráculo, no añade nada sobre M1.

**5.3 Cómo debe leerse.**

Tiempo de ajuste+predicción (orden minutos en CPU) y matriz de predicciones duras lista para el reporte por técnica.

**5.4 Trazabilidad.**

Sus salidas alimentan todas las métricas de la sección 6. La complejidad es O(L · costo-LogReg) con L=6 etiquetas.



In [4]:
# modelo: OneVsRest LogReg balanceado + baseline oraculo-tier
import time
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

t0 = time.time()
clf = OneVsRestClassifier(LogisticRegression(max_iter=1000, C=0.5, class_weight='balanced'))
clf.fit(Xtr, Y[tr])
P = clf.predict(Xte)
print(f'fit+predict: {time.time()-t0:.0f}s')

# oraculo-tier: suspicious->[T1190], malicious->[T1190,T1041,T1567] (+T1204 impredecible)
oracle = np.zeros_like(Y[te])
lab_te = th[te].label_binary.to_numpy()
oracle[:, TECHS.index('T1190')] = 1
mal = lab_te == 'malicious'
oracle[mal, TECHS.index('T1041')] = 1
oracle[mal, TECHS.index('T1567')] = 1
print('oraculo listo')

C:\Users\nicor\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


C:\Users\nicor\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


fit+predict: 56s
oraculo listo


C:\Users\nicor\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 4 is present in all training examples.
  warnings.warn(
C:\Users\nicor\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 5 is present in all training examples.
  warnings.warn(


#### 5.1 Observaciones e insights

**Observación.**

Modelo ajustado en minutos CPU; predicciones duras y oráculo-tier construidos sobre el mismo test.

**Interpretación.**

El oráculo es casi perfecto por construcción (micro 0.9994): confirma que el tier determina las técnicas en esta captura, no al revés. El modelo lineal parte con desventaja estructural: debe redescubrir desde X lo que el oráculo recibe gratis (el tier real).

**Conexión.**

Anticipa el veredicto: el modelo solo será útil si separa T1041/T1567 dentro del threat mejor que el azar informado.



### 6. Evaluación: reporte por técnica, micro/macro y PR-AUC

**6.1 Qué se hace.**

Reporte de clasificación por técnica, micro-F1, macro-F1, hamming, subset accuracy (modelo y oráculo) y PR-AUC por técnica con soporte en test.

**6.2 Por qué.**

El par micro/macro es obligatorio: el micro lo domina T1190 (constante) y el macro castiga cabezas fallidas y cola ausente. El subset accuracy mide exactitud de conjunto exacto (criterio más duro, propio de atribución). El oráculo calibra la escala 0–techo.

**6.3 Cómo debe leerse.**

Por técnica: precisión/recall/F1 y soporte. Global: micro vs macro (la brecha es el diagnóstico), hamming (trivialmente bajo) y subset-acc frente al oráculo. Nota metrológica: el PR-AUC aquí se computa sobre predicciones duras; el protocolo correcto usa scores continuos y queda como corrección pendiente en la familia M4 (ver M4d).

**6.4 Trazabilidad.**

Produce los números que persiste la sección 7 y que cita Limitations del artículo.



In [5]:
from sklearn.metrics import (classification_report, f1_score, hamming_loss,
                             accuracy_score, average_precision_score)
print('=== M4 OneVsRest-LogReg ===')
print(classification_report(Y[te], P, target_names=TECHS, zero_division=0))
print('micro-F1:', round(f1_score(Y[te], P, average='micro'), 4),
      '| macro-F1:', round(f1_score(Y[te], P, average='macro'), 4))
print('hamming:', round(hamming_loss(Y[te], P), 4),
      '| subset-acc:', round(accuracy_score(Y[te], P), 4))
print('=== baseline oraculo-tier ===')
print('micro-F1:', round(f1_score(Y[te], oracle, average='micro'), 4),
      '| macro-F1:', round(f1_score(Y[te], oracle, average='macro'), 4),
      '| hamming:', round(hamming_loss(Y[te], oracle), 4),
      '| subset-acc:', round(accuracy_score(Y[te], oracle), 4))
for j, t in enumerate(TECHS):
    try:
        ap = average_precision_score(Y[te][:, j], P[:, j])
    except Exception:
        ap = 0.0
    print(f'{t}: soporte test {int(Y[te][:, j].sum())}  PR-AUC {round(float(ap), 4)}')

=== M4 OneVsRest-LogReg ===
              precision    recall  f1-score   support

       T1190       1.00      1.00      1.00     22892
       T1041       0.08      0.12      0.10      3733
       T1567       0.08      0.11      0.10      3721
       T1204       0.19      1.00      0.32        10
       T1486       0.00      0.00      0.00         0
       T1566       0.00      0.00      0.00         0

   micro avg       0.72      0.78      0.75     30356
   macro avg       0.23      0.37      0.25     30356
weighted avg       0.77      0.78      0.78     30356
 samples avg       0.86      0.90      0.82     30356

micro-F1: 0.7484 | macro-F1: 0.2516
hamming: 0.1162 | subset-acc: 0.6451
=== baseline oraculo-tier ===
micro-F1: 0.9994 | macro-F1: 0.4997 | hamming: 0.0002 | subset-acc: 0.999
T1190: soporte test 22892  PR-AUC 1.0
T1041: soporte test 3733  PR-AUC 0.1537
T1567: soporte test 3721  PR-AUC 0.1537
T1204: soporte test 10  PR-AUC 0.1887
T1486: soporte test 0  PR-AUC 0.0
T1566: s

C:\Users\nicor\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\nicor\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\nicor\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
C:\Users\nicor\AppData\Local\Programs\Python\Python310\lib\site-p

#### 6.1 Observaciones e insights

**Observación.**

M4: micro-F1 0.7484, macro-F1 0.2516, hamming 0.1162, subset-acc 0.6451 (oráculo 0.9994/0.4997). T1190 1.00/1.00/1.00 trivial; T1041/T1567 precisión 0.08, recall 0.11–0.12, F1 0.10, PR-AUC 0.15; T1204 recall 1.00 (10/10) con precisión 0.19; T1486/T1566 F1 0 sin soporte.

**Interpretación.**

H4a verificada: el lineal convergido ni dispara en T1041/T1567; no separa malicioso de sospechoso a nivel evento con estos features. H4b verificada: la cola no es aprendible por escasez. El micro alto es espejismo de T1190; la brecha micro–macro (0.50) es el diagnóstico honesto.

**Conexión.**

Veredicto para la demo: no prometer ATT&CK por evento. La consola vende cabeza T1190 y documenta el resto; la cola pide few-shot o LLM con `incident_signals` histórico o reglas. M4b/M4c confirman el techo con XGB.



### 7. Persistencia del experimento

**7.1 Qué se hace.**

Se escribe `models/m4_metrics.json` con etiquetas, micro/macro-F1, hamming, subset-acc, tamaños y el par del oráculo.

**7.2 Por qué.**

Sin artefacto versionado no hay ablación auditable: el artículo cita estos números en Limitations y la consola decide con ellos qué servir como hints.

**7.3 Cómo debe leerse.**

Claves del JSON y correspondencia con lo impreso en la sección 6.

**7.4 Trazabilidad.**

Cierra el cuaderno hacia `models/m4b_metrics.json` y `models/m4c_exps.json` (misma familia, mismo protocolo).



In [6]:
import json
m = {'model': 'm4-multilabel-v1', 'labels': TECHS,
     'micro_f1': round(float(f1_score(Y[te], P, average='micro')), 4),
     'macro_f1': round(float(f1_score(Y[te], P, average='macro')), 4),
     'hamming': round(float(hamming_loss(Y[te], P)), 4),
     'subset_acc': round(float(accuracy_score(Y[te], P)), 4),
     'n_train': int(tr.sum()), 'n_test': int(te.sum()),
     'oracle': {'micro_f1': round(float(f1_score(Y[te], oracle, average='micro')), 4),
                'macro_f1': round(float(f1_score(Y[te], oracle, average='macro')), 4)}}
(ROOT / 'models' / 'm4_metrics.json').write_text(json.dumps(m, indent=2))
print('guardado models/m4_metrics.json', m)

guardado models/m4_metrics.json {'model': 'm4-multilabel-v1', 'labels': ['T1190', 'T1041', 'T1567', 'T1204', 'T1486', 'T1566'], 'micro_f1': 0.7484, 'macro_f1': 0.2516, 'hamming': 0.1162, 'subset_acc': 0.6451, 'n_train': 91617, 'n_test': 22904, 'oracle': {'micro_f1': 0.9994, 'macro_f1': 0.4997}}


C:\Users\nicor\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\nicor\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


#### 7.1 Observaciones e insights

**Observación.**

`m4_metrics.json` persistido con micro 0.7484, macro 0.2516 y oráculo 0.9994/0.4997; n_train 91 617, n_test 22 904.

**Interpretación.**

El experimento queda reproducible y comparable dentro de la familia M4: mismos 6 labels, mismo corte, mismo X. La trazabilidad permite afirmar el techo sin reejecutar.

**Conexión.**

Puente a M4b (XGB por técnica) y M4c (tabular+grafo): misma pregunta, más capacidad y más contexto.



### 8. Conclusión del cuaderno: la atribución por evento tiene techo

#### 8.1 Resultados consolidados

| Contraste | Hallazgo | Valor |
|---|---|---|
| Cabeza trivial | T1190 perfecto por prevalencia | P/R/F1 1.00, no mide nada |
| Cabeza dura | T1041/T1567 con lineal | P 0.08, F1 0.10, PR-AUC 0.15 |
| Cola | T1204 a ciegas; T1486/T1566 sin soporte | R 1.00 con P 0.19; F1 0 |
| Global M4 | micro vs macro | 0.7484 vs 0.2516 (la brecha es el diagnóstico) |
| Techo | oráculo-tier | micro 0.9994, macro 0.4997 |

#### 8.2 Cadena de hallazgos

- **El tier determina las técnicas, no al revés.** El oráculo casi perfecto lo demuestra: predecir técnicas desde el tier es trivial porque la correlación Precinct las asigna por regla.

- **El lineal no separa cabezas dentro del threat.** T1041/T1567 exigen distinguir malicioso de sospechoso a nivel evento, y con TF-IDF + tabular el lineal ni dispara.

- **La cola es escasez, no capacidad.** Sin soporte en live no hay aprendizaje posible; más árboles (M4b) o más contexto (M4c) solo confirman el techo.

#### 8.3 Plan resultante

**Y** multilabel sobre threat con X de M1/M2 reutilizado. Métricas obligatorias en pareja micro/macro más subset-acc. Corrección pendiente: PR-AUC sobre scores continuos en la familia M4.

#### 8.4 Paso siguiente

`notebooks/m4b_xgb_pertecnica.ipynb` (misma pregunta con XGBoost por técnica) y luego M4c (tabular+grafo). Veredicto ya fijado: hints tier→técnicas en consola, sin modelo de atribución desplegado.

*Medir el techo con honestidad es lo que autoriza a no desplegarlo.*

